# MCQ Ensemble Pipeline — DeBERTa + RoBERTa
Fill in the blank cells marked `# <<< FILL IN >>>` with your paths/column names, then run all cells top to bottom.

In [ ]:
# ================= SETUP =================
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABELS = ['A', 'B', 'C', 'D', 'E']


In [ ]:
# ================= PATHS & DATA (FILL IN) =================
DEBERTA_CKPT = ""   # <<< FILL IN >>> e.g. "microsoft/deberta-v3-small" fine-tuned checkpoint path or HF repo
ROBERTA_CKPT = ""   # <<< FILL IN >>> e.g. "roberta-base" fine-tuned checkpoint path or HF repo

TRAIN_CSV = ""       # <<< FILL IN >>> path to train.csv
TEST_CSV  = ""       # <<< FILL IN >>> path to test.csv

# Column names in your csv
ID_COL       = ""    # <<< FILL IN >>> e.g. "id"
PROMPT_COL   = ""    # <<< FILL IN >>> e.g. "prompt"
OPTION_COLS  = []     # <<< FILL IN >>> e.g. ["A","B","C","D","E"]
ANSWER_COL   = ""    # <<< FILL IN, only needed for MAP@3 (Q10), e.g. "answer" >>>

train_df = pd.read_csv(TRAIN_CSV) if TRAIN_CSV else None
test_df  = pd.read_csv(TEST_CSV) if TEST_CSV else None
test_df.head() if test_df is not None else None


In [ ]:
# ================= LOAD MODELS =================
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_CKPT)
deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_CKPT, num_labels=5).to(device)
deberta_model.eval()

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_CKPT)
roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_CKPT, num_labels=5).to(device)
roberta_model.eval()

print("Models loaded successfully.")


In [ ]:
# ================= HELPER FUNCTIONS =================

def build_input_text(row, prompt_col=PROMPT_COL, option_cols=OPTION_COLS):
    """Combine prompt + options into a single text string for the classifier.
    Adjust this if your fine-tuning format differs (e.g. separate per-option encoding).
    """
    prompt = str(row[prompt_col])
    options_text = "\n".join([f"{LABELS[i]}. {row[c]}" for i, c in enumerate(option_cols)])
    return f"{prompt}\n{options_text}"


@torch.no_grad()
def predict_proba(text, tokenizer, model, max_length=512):
    """Run a single text through a model, return softmax probabilities (len-5 numpy array)."""
    enc = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=max_length).to(device)
    logits = model(**enc).logits.squeeze(0)
    probs = F.softmax(logits, dim=-1).cpu().numpy()
    return probs


def row_text(df, idx, prompt_col=PROMPT_COL, option_cols=OPTION_COLS, prefix=""):
    row = df.iloc[idx]
    text = build_input_text(row, prompt_col, option_cols)
    return prefix + text


def top1(probs):
    idx = int(np.argmax(probs))
    return LABELS[idx], float(probs[idx])


def top3_string(probs):
    order = np.argsort(-probs)[:3]
    return " ".join(LABELS[i] for i in order)


def weighted_ensemble(p_deberta, p_roberta, w_deberta=0.7, w_roberta=0.3):
    return w_deberta * p_deberta + w_roberta * p_roberta


def simple_ensemble(p_deberta, p_roberta):
    return (p_deberta + p_roberta) / 2.0


## Question 1 — DeBERTa top prediction, row 25

In [ ]:
ROW_IDX = 25

text_25 = row_text(test_df, ROW_IDX)
p_deberta_25 = predict_proba(text_25, deberta_tokenizer, deberta_model)
p_roberta_25 = predict_proba(text_25, roberta_tokenizer, roberta_model)

q1_label, q1_prob = top1(p_deberta_25)
print(f"Q1 -> {q1_label}, probability of {q1_label} = {q1_prob:.4f}")


## Question 2 — Simple average ensembling, row 25

In [ ]:
p_simple_25 = simple_ensemble(p_deberta_25, p_roberta_25)
q2_label, q2_prob = top1(p_simple_25)
print(f"Q2 -> {q2_label}, probability = {q2_prob:.4f}")


## Question 3 — Weighted ensembling (0.7/0.3), row 25

In [ ]:
p_weighted_25 = weighted_ensemble(p_deberta_25, p_roberta_25, 0.7, 0.3)
q3_label, q3_prob = top1(p_weighted_25)
print(f"Q3 -> {q3_label}, probability = {q3_prob:.4f}")


## Question 4 — Top-3 string, row 25

In [ ]:
q4_top3 = top3_string(p_weighted_25)
print(f"Q4 -> Top-3 for row {ROW_IDX}: {q4_top3}")


## Question 5 — Full weighted-ensemble pipeline on test.csv -> submission.csv

In [ ]:
def run_full_pipeline(df):
    rows = []
    for i in range(len(df)):
        text = row_text(df, i)
        p_d = predict_proba(text, deberta_tokenizer, deberta_model)
        p_r = predict_proba(text, roberta_tokenizer, roberta_model)
        p_w = weighted_ensemble(p_d, p_r, 0.7, 0.3)
        rows.append({
            'id': df.iloc[i][ID_COL],
            'prediction': top3_string(p_w)
        })
    return pd.DataFrame(rows)

submission_df = run_full_pipeline(test_df)
submission_df.to_csv('submission.csv', index=False)

q5_count = len(submission_df)
print(f"Q5 -> Number of prediction rows in submission.csv: {q5_count}")
submission_df.head()


## Question 6 — Test-Time Augmentation on first 50 rows (DeBERTa only)

In [ ]:
TTA_PREFIX = "Answer the following multiple-choice question carefully: "
N_TTA = 50

tta_diff_count = 0
tta_details = []

for i in range(min(N_TTA, len(test_df))):
    text_orig = row_text(test_df, i)
    text_aug  = row_text(test_df, i, prefix=TTA_PREFIX)

    p_orig = predict_proba(text_orig, deberta_tokenizer, deberta_model)
    p_aug  = predict_proba(text_aug, deberta_tokenizer, deberta_model)

    p_tta_avg = (p_orig + p_aug) / 2.0

    label_orig, _ = top1(p_orig)
    label_tta, _  = top1(p_tta_avg)

    is_diff = label_orig != label_tta
    tta_diff_count += int(is_diff)
    tta_details.append({'row': i, 'orig_top1': label_orig, 'tta_top1': label_tta, 'changed': is_diff})

q6_count = tta_diff_count
print(f"Q6 -> Rows with different Top-1 after TTA: {q6_count} / {N_TTA}")
pd.DataFrame(tta_details).head()


## Question 7 — DeBERTa vs Weighted Ensemble Top-1, first 100 rows

In [ ]:
N_CMP = 100

records = []
for i in range(min(N_CMP, len(test_df))):
    text = row_text(test_df, i)
    p_d = predict_proba(text, deberta_tokenizer, deberta_model)
    p_r = predict_proba(text, roberta_tokenizer, roberta_model)
    p_w = weighted_ensemble(p_d, p_r, 0.7, 0.3)

    d_label, d_conf = top1(p_d)
    w_label, w_conf = top1(p_w)

    records.append({
        'row': i,
        'deberta_top1': d_label, 'deberta_conf': d_conf,
        'ensemble_top1': w_label, 'ensemble_conf': w_conf,
        'deberta_top3': top3_string(p_d),
        'ensemble_top3': top3_string(p_w),
    })

cmp_df = pd.DataFrame(records)

q7_count = (cmp_df['deberta_top1'] != cmp_df['ensemble_top1']).sum()
print(f"Q7 -> Rows with different Top-1 (DeBERTa vs Ensemble): {q7_count} / {N_CMP}")
cmp_df.head()


## Question 8 — Confidence gain (Ensemble - DeBERTa), first 100 rows

In [ ]:
cmp_df['confidence_gain'] = cmp_df['ensemble_conf'] - cmp_df['deberta_conf']
q8_count = (cmp_df['confidence_gain'] > 0).sum()
print(f"Q8 -> Rows with positive confidence gain: {q8_count} / {N_CMP}")


## Question 9 — Top-3 ranking changes, first 100 rows

In [ ]:
q9_count = (cmp_df['deberta_top3'] != cmp_df['ensemble_top3']).sum()
print(f"Q9 -> Rows with at least one Top-3 ranking change: {q9_count} / {N_CMP}")


## Question 10 — MAP@3 on first 100 rows (requires ground-truth ANSWER_COL)

In [ ]:
def average_precision_at_3(pred_labels, true_label):
    """pred_labels: list of up to 3 predicted labels in ranked order. true_label: single correct label."""
    for i, p in enumerate(pred_labels[:3]):
        if p == true_label:
            return 1.0 / (i + 1)
    return 0.0

assert ANSWER_COL, "Set ANSWER_COL to your ground-truth column name to compute MAP@3"

ap_scores = []
for i in range(min(N_CMP, len(test_df))):
    true_label = str(test_df.iloc[i][ANSWER_COL]).strip()
    pred_top3 = cmp_df.iloc[i]['ensemble_top3'].split()
    ap_scores.append(average_precision_at_3(pred_top3, true_label))

q10_map3 = round(float(np.mean(ap_scores)), 4)
print(f"Q10 -> MAP@3 = {q10_map3}")


## Summary of all answers

In [ ]:
print("Q1:", q1_label, q1_prob)
print("Q2:", q2_label, q2_prob)
print("Q3:", q3_label, q3_prob)
print("Q4:", q4_top3)
print("Q5:", q5_count)
print("Q6:", q6_count)
print("Q7:", q7_count)
print("Q8:", q8_count)
print("Q9:", q9_count)
print("Q10:", q10_map3)
